# Week-6 Spark Assignment
**Internship:** Celebal Technologies | Data Engineering Track  
**Topic:** Apache Spark — Architecture, Lazy Evaluation, Transformations, File Formats  


## Setup — SparkSession Initialization

In [1]:
import os, warnings
warnings.filterwarnings("ignore")
os.environ["PYSPARK_PYTHON"] = "python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python3"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

spark = (SparkSession.builder
         .appName("Week6_Spark_Assignment")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "4")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession started successfully.")
print(f"Spark Version: {spark.version}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/28 11:58:19 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/06/28 11:58:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/28 11:58:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession started successfully.
Spark Version: 4.1.2


---
## Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

In a Spark application, three main components work together to execute distributed workloads:

**Driver:**
- The Driver is the process that runs the `main()` function of the Spark application.
- It creates the `SparkContext` / `SparkSession`, which is the entry point to all Spark functionality.
- It converts the user's code into a **DAG (Directed Acyclic Graph)** of stages and tasks.
- It communicates with the Cluster Manager to request resources, and schedules tasks on Executors.
- It collects results from Executors when actions like `collect()` or `show()` are called.

**Cluster Manager:**
- The Cluster Manager is responsible for **resource allocation** across the cluster.
- It tracks available machines (nodes), their memory and CPU, and allocates them to Spark applications.
- Supported Cluster Managers: **Standalone** (built-in), **Apache YARN**, **Apache Mesos**, **Kubernetes**.
- In local mode (development), Spark acts as its own Cluster Manager.

**Executor:**
- Executors are **JVM processes** launched on worker nodes by the Cluster Manager.
- Each Executor runs **tasks** assigned by the Driver and stores intermediate data in memory/disk.
- Executors report task status and results back to the Driver.
- They live for the entire lifetime of the application.

```
User Code
    │
    ▼
 [Driver]  ──── schedules tasks ──▶  [Executor 1]
    │                                [Executor 2]
    ▼                                [Executor N]
[Cluster Manager]
(allocates resources)
```


---
## Q2: How does Spark's Lazy Evaluation strategy improve performance when chain-processing large datasets?

**Lazy Evaluation** means Spark does **not execute transformations immediately**. Instead, it records each transformation in a DAG and only runs the computation when an **action** (e.g., `show()`, `count()`, `write()`) is called.

**How it improves performance:**

1. **Optimization before execution:** The Catalyst optimizer analyzes the full DAG and rewrites it into the most efficient physical plan — e.g., reordering filters, eliminating unused columns.

2. **Predicate Pushdown:** Filters are automatically moved as early as possible in the plan, so less data is read and shuffled.

3. **Pipelining:** Multiple narrow transformations (e.g., `filter → select → withColumn`) are combined into a single stage and executed in one pass over the data — no intermediate writes to disk.

4. **Avoid unnecessary computation:** If only the first 5 rows are needed (`show(5)`), Spark can stop after finding them — it doesn't process the whole dataset.

**Example:** In the chain below, Spark optimizes all three steps into a single scan:
```python
df.filter(...).select(...).withColumn(...)  # nothing runs yet
   .show()  # NOW Spark executes — optimized single pass
```


In [2]:
# Q2 — Demonstrating Lazy Evaluation

# Create a sample DataFrame
data = [(1, "Electronics", 250.0),
        (2, "Clothing",    89.99),
        (3, "Electronics", 1200.0),
        (4, "Furniture",   450.0),
        (5, "Electronics", 30.0)]

df_demo = spark.createDataFrame(data, ["product_id", "category", "price"])

# These transformations are LAZY — nothing executes yet
transformed = (df_demo
               .filter(F.col("category") == "Electronics")
               .withColumn("price_with_tax", F.col("price") * 1.18)
               .select("product_id", "category", "price_with_tax"))

print("Transformations registered (lazy) — no execution yet.")
print("\nOptimized Physical Plan:")
transformed.explain(mode="simple")

# Action triggers execution
print("\nAction called — now Spark executes:")
transformed.show()


Transformations registered (lazy) — no execution yet.

Optimized Physical Plan:


== Physical Plan ==
*(1) Project [product_id#0L, category#1, (price#2 * 1.18) AS price_with_tax#3]
+- *(1) Filter (isnotnull(category#1) AND (category#1 = Electronics))
   +- *(1) Scan ExistingRDD[product_id#0L,category#1,price#2]



Action called — now Spark executes:


+----------+-----------+--------------+
|product_id|   category|price_with_tax|
+----------+-----------+--------------+
|         1|Electronics|         295.0|
|         3|Electronics|        1416.0|
|         5|Electronics|          35.4|
+----------+-----------+--------------+



---
## Q3: Write a Spark command to read a CSV file located at `"data/source.csv"`, ensuring the first row is treated as a header and `inferSchema` is enabled.


In [3]:
# Q3 — Reading a CSV file with header and inferSchema

# First, create a sample CSV file to demonstrate
import os
os.makedirs("data", exist_ok=True)
with open("data/source.csv", "w") as f:
    f.write("product_id,category,price,quantity\n")
    f.write("101,Electronics,999.99,5\n")
    f.write("102,Clothing,49.99,20\n")
    f.write("103,Furniture,299.00,8\n")
    f.write("104,Electronics,1499.50,3\n")

# The required Spark command
df = (spark.read
      .option("header", "true")       # first row treated as column names
      .option("inferSchema", "true")  # Spark auto-detects data types
      .csv("data/source.csv"))

print("CSV loaded successfully.")
print("\nInferred Schema:")
df.printSchema()

print("Data:")
df.show()


CSV loaded successfully.

Inferred Schema:
root
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)

Data:


+----------+-----------+------+--------+
|product_id|   category| price|quantity|
+----------+-----------+------+--------+
|       101|Electronics|999.99|       5|
|       102|   Clothing| 49.99|      20|
|       103|  Furniture| 299.0|       8|
|       104|Electronics|1499.5|       3|
+----------+-----------+------+--------+



---
## Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

| Feature | CSV (Row-based) | Parquet (Columnar) |
|---|---|---|
| **Storage layout** | Data stored row by row | Data stored column by column |
| **Compression** | Minimal (plain text) | High — similar values in a column compress very well |
| **Schema** | No embedded schema — must be inferred or declared each time | Schema embedded in file metadata |
| **Column pruning** | Must read entire row even if only 1 column needed | Reads **only the requested columns** from disk |
| **Predicate pushdown** | Not supported | Supported — skips entire row groups using min/max stats |
| **Read performance** | Slow for analytical queries | Very fast for OLAP/aggregation workloads |
| **Write performance** | Fast | Slightly slower (encoding + compression) |
| **Human readable** | Yes | No (binary format) |

**Why it matters for performance:**

In analytical workloads, queries typically access only a **few columns** out of many. With CSV, Spark must read and parse every column in every row. With Parquet, only the required columns are read from disk — this can reduce I/O by 90%+ on wide tables.

Additionally, Parquet's **row group statistics** (min/max per column) allow Spark to skip entire chunks of data that can't match a filter, without reading them at all.


In [4]:
# Q4 — CSV vs Parquet: size and read performance comparison
import shutil, time

# Write same data as Parquet
PARQUET_PATH = "/tmp/source_parquet"
CSV_PATH_OUT = "/tmp/source_csv_out"

for p in [PARQUET_PATH, CSV_PATH_OUT]:
    if os.path.exists(p): shutil.rmtree(p)

# Write CSV
t0 = time.time()
df.write.mode("overwrite").option("header","true").csv(CSV_PATH_OUT)
csv_write = time.time() - t0

# Write Parquet
t0 = time.time()
df.write.mode("overwrite").parquet(PARQUET_PATH)
parquet_write = time.time() - t0

# File sizes
def dir_kb(path):
    total = 0
    for r, _, files in os.walk(path):
        for f in files:
            if not f.startswith("."):
                total += os.path.getsize(os.path.join(r, f))
    return round(total / 1024, 2)

print("=" * 45)
print("     CSV vs Parquet Comparison")
print("=" * 45)
print(f"  CSV    write time : {csv_write:.3f}s  | size: {dir_kb(CSV_PATH_OUT)} KB")
print(f"  Parquet write time: {parquet_write:.3f}s  | size: {dir_kb(PARQUET_PATH)} KB")
print("=" * 45)
print("Parquet uses columnar storage + snappy compression")
print("→ Smaller files + faster analytical reads")


     CSV vs Parquet Comparison
  CSV    write time : 0.744s  | size: 0.13 KB
  Parquet write time: 1.428s  | size: 1.32 KB
Parquet uses columnar storage + snappy compression
→ Smaller files + faster analytical reads


---
## Q5: Given a DataFrame `df`, write a query to select the columns `product_id` and `price` where the `category` is 'Electronics'.


In [5]:
# Q5 — Select product_id and price where category is 'Electronics'

# Using the df loaded from data/source.csv in Q3
result_q5 = (df
             .filter(F.col("category") == "Electronics")
             .select("product_id", "price"))

print("product_id and price where category = 'Electronics':")
result_q5.show()


product_id and price where category = 'Electronics':


+----------+------+
|product_id| price|
+----------+------+
|       101|999.99|
|       104|1499.5|
+----------+------+



---
## Q6: Write the code to "revise" a DataFrame by renaming the column `old_name` to `new_name` and casting the `price` column from a String to a Double.


In [6]:
# Q6 — Rename column and cast price to Double

# Create a sample DataFrame where price is a String (simulating raw CSV read without inferSchema)
data_q6 = [("P001", "Laptop",  "999.99"),
            ("P002", "Monitor", "349.50"),
            ("P003", "Keyboard","49.00")]

df_q6 = spark.createDataFrame(data_q6, ["product_id", "old_name", "price"])

print("Original schema (price is String):")
df_q6.printSchema()

# Rename old_name → new_name, cast price String → Double
df_revised = (df_q6
              .withColumnRenamed("old_name", "new_name")
              .withColumn("price", F.col("price").cast(DoubleType())))

print("Revised schema (price is Double, column renamed):")
df_revised.printSchema()
df_revised.show()


Original schema (price is String):
root
 |-- product_id: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- price: string (nullable = true)

Revised schema (price is Double, column renamed):
root
 |-- product_id: string (nullable = true)
 |-- new_name: string (nullable = true)
 |-- price: double (nullable = true)



+----------+--------+------+
|product_id|new_name| price|
+----------+--------+------+
|      P001|  Laptop|999.99|
|      P002| Monitor| 349.5|
|      P003|Keyboard|  49.0|
+----------+--------+------+



---
## Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

Spark achieves **fault tolerance without replication** using the **DAG (Directed Acyclic Graph)** — also called the **lineage graph**.

**How it works:**

1. Every transformation applied to an RDD or DataFrame is recorded as a step in the DAG.
2. The DAG captures the full **history of how data was derived** from the original source.
3. If a worker node fails and a partition of data is lost, Spark does **not crash**.
4. Instead, it **re-computes only the lost partitions** by replaying the relevant steps from the DAG — starting from the original data source or the last checkpoint.

**Example:**
```
Source CSV
    │
    ▼  filter(category == "Electronics")
    │
    ▼  withColumn(final_price = price * 1.18)
    │
    ▼  groupBy(category).agg(sum(final_price))
         [Executor 3 FAILS here]
         → Spark re-runs only the lost partition of Executor 3
           using the recorded lineage
```

**Key benefits:**
- No need to replicate data across nodes (unlike Hadoop's 3x replication)
- Only the **affected partitions** are recomputed, not the whole dataset
- Works transparently — the application doesn't need to handle failures manually

**Checkpointing:** For very long lineage chains, `df.checkpoint()` can be used to save intermediate state to disk, so re-computation starts from the checkpoint instead of the beginning.


---
## Q8: Write a query to filter a DataFrame `df_orders` for rows where the `status` is 'Completed' AND the `amount` is greater than 1000.


In [7]:
# Q8 — Filter df_orders: status = 'Completed' AND amount > 1000

data_q8 = [(1, "Completed", 1500.0),
            (2, "Pending",   800.0),
            (3, "Completed", 500.0),
            (4, "Completed", 2300.0),
            (5, "Cancelled", 1100.0),
            (6, "Completed", 999.0),
            (7, "Completed", 1050.0)]

df_orders = spark.createDataFrame(data_q8, ["order_id", "status", "amount"])

print("Full df_orders:")
df_orders.show()

# Filter: status == 'Completed' AND amount > 1000
filtered_orders = df_orders.filter(
    (F.col("status") == "Completed") & (F.col("amount") > 1000)
)

print("Filtered — status='Completed' AND amount > 1000:")
filtered_orders.show()


Full df_orders:


+--------+---------+------+
|order_id|   status|amount|
+--------+---------+------+
|       1|Completed|1500.0|
|       2|  Pending| 800.0|
|       3|Completed| 500.0|
|       4|Completed|2300.0|
|       5|Cancelled|1100.0|
|       6|Completed| 999.0|
|       7|Completed|1050.0|
+--------+---------+------+

Filtered — status='Completed' AND amount > 1000:


+--------+---------+------+
|order_id|   status|amount|
+--------+---------+------+
|       1|Completed|1500.0|
|       4|Completed|2300.0|
|       7|Completed|1050.0|
+--------+---------+------+



---
## Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

**Predicate Pushdown** is an optimization where Spark (via the Catalyst optimizer) **pushes filter conditions down to the data source level**, so that data is filtered **before** being loaded into memory — rather than after.

**How it works with Parquet:**

Parquet stores data in **row groups**, and for each column in each row group it stores **statistics** (minimum value, maximum value, null count). When Spark needs to apply a filter like `age > 30`, it:

1. Reads only the row group **metadata** (very small)
2. Checks if the row group's `max(age) <= 30` → if yes, **skip the entire row group** without reading it
3. Only loads row groups that **could** contain matching rows

**Impact on memory:**
- Without pushdown: entire dataset loaded → filtered in memory
- With pushdown: only matching row groups loaded → far less data in memory

**Example:**
```python
# Spark automatically pushes this filter to the Parquet reader
df = spark.read.parquet("data/orders.parquet")
df.filter(F.col("amount") > 1000).show()
# → Parquet reader skips row groups where max(amount) <= 1000
#   before any data reaches Spark's memory
```

This is one of the primary reasons Parquet outperforms CSV for analytical workloads — CSV has no metadata, so every byte must be read and then filtered in memory.


In [8]:
# Q9 — Demonstrating Predicate Pushdown on Parquet

import shutil
PRED_PATH = "/tmp/pred_pushdown_demo"
if os.path.exists(PRED_PATH): shutil.rmtree(PRED_PATH)

# Write a Parquet file
df_orders.write.mode("overwrite").parquet(PRED_PATH)

# Read back and apply filter — Catalyst will push it down to Parquet reader
df_parquet_read = spark.read.parquet(PRED_PATH)
df_filtered = df_parquet_read.filter(F.col("amount") > 1000)

print("Physical Plan — observe 'PushedFilters' in the scan:")
df_filtered.explain(mode="simple")
df_filtered.show()
print("The filter 'amount > 1000' is pushed to Parquet reader level.")
print("Only matching row groups are loaded into Spark memory.")


Physical Plan — observe 'PushedFilters' in the scan:
== Physical Plan ==
*(1) Filter (isnotnull(amount#111) AND (amount#111 > 1000.0))
+- *(1) ColumnarToRow
   +- FileScan parquet [order_id#109L,status#110,amount#111] Batched: true, DataFilters: [isnotnull(amount#111), (amount#111 > 1000.0)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/pred_pushdown_demo], PartitionFilters: [], PushedFilters: [IsNotNull(amount), GreaterThan(amount,1000.0)], ReadSchema: struct<order_id:bigint,status:string,amount:double>




+--------+---------+------+
|order_id|   status|amount|
+--------+---------+------+
|       1|Completed|1500.0|
|       4|Completed|2300.0|
|       5|Cancelled|1100.0|
|       7|Completed|1050.0|
+--------+---------+------+

The filter 'amount > 1000' is pushed to Parquet reader level.
Only matching row groups are loaded into Spark memory.


---
## Q10: Write a code snippet to add a new column `final_price` which is the `base_price` multiplied by 1.18 (18% tax).


In [9]:
# Q10 — Add final_price = base_price * 1.18 (18% GST)

data_q10 = [("Laptop",  45000.0),
             ("Phone",   15000.0),
             ("Tablet",  25000.0),
             ("Monitor", 18000.0)]

df_q10 = spark.createDataFrame(data_q10, ["product", "base_price"])

# Add new column final_price = base_price * 1.18
df_with_tax = df_q10.withColumn(
    "final_price",
    F.round(F.col("base_price") * 1.18, 2)
)

print("DataFrame with final_price (base_price × 1.18 tax):")
df_with_tax.show()


DataFrame with final_price (base_price × 1.18 tax):


+-------+----------+-----------+
|product|base_price|final_price|
+-------+----------+-----------+
| Laptop|   45000.0|    53100.0|
|  Phone|   15000.0|    17700.0|
| Tablet|   25000.0|    29500.0|
|Monitor|   18000.0|    21240.0|
+-------+----------+-----------+



---
## Q11: What is the difference between Transformations and Actions? Provide two examples of each.

| | **Transformations** | **Actions** |
|---|---|---|
| **Definition** | Operations that define a new DataFrame from an existing one | Operations that trigger execution and return a result |
| **Execution** | Lazy — recorded in DAG, not executed immediately | Eager — immediately triggers DAG execution |
| **Return type** | Returns a new DataFrame | Returns a value, Row, or writes output |
| **Examples** | `filter()`, `select()`, `withColumn()`, `groupBy()`, `join()` | `show()`, `count()`, `collect()`, `write()`, `first()` |

**Transformations — Two Examples:**
1. `filter(F.col("price") > 100)` — narrows down rows (lazy)
2. `withColumn("tax", F.col("price") * 0.18)` — adds a new column (lazy)

**Actions — Two Examples:**
1. `count()` — returns the number of rows (triggers execution)
2. `show(5)` — fetches and displays the first 5 rows (triggers execution)


In [10]:
# Q11 — Transformations vs Actions demonstration

df_q11 = spark.createDataFrame(
    [(1, "A", 200.0), (2, "B", 50.0), (3, "A", 800.0), (4, "C", 120.0)],
    ["id", "category", "price"]
)

# TRANSFORMATIONS (lazy — no execution)
t1 = df_q11.filter(F.col("price") > 100)                        # Transformation 1
t2 = t1.withColumn("discounted", F.col("price") * 0.9)          # Transformation 2
print("Transformations defined (lazy — nothing executed yet)")

# ACTIONS (trigger execution)
print("\nAction 1 — count():", t2.count())                      # Action 1
print("\nAction 2 — show():")
t2.show()                                                         # Action 2


Transformations defined (lazy — nothing executed yet)



Action 1 — count(): 3

Action 2 — show():


+---+--------+-----+----------+
| id|category|price|discounted|
+---+--------+-----+----------+
|  1|       A|200.0|     180.0|
|  3|       A|800.0|     720.0|
|  4|       C|120.0|     108.0|
+---+--------+-----+----------+



---
## Q12: Write the Spark command to load a Parquet file from `"path/to/input"`, filter out any rows where `user_id` is null, and save the result as a CSV at `"path/to/output"`.


In [11]:
# Q12 — Load Parquet, filter nulls, save as CSV

import shutil

INPUT_PATH  = "/tmp/path_to_input"
OUTPUT_PATH = "/tmp/path_to_output"

# Setup: create sample Parquet with some null user_ids
if os.path.exists(INPUT_PATH): shutil.rmtree(INPUT_PATH)
if os.path.exists(OUTPUT_PATH): shutil.rmtree(OUTPUT_PATH)

data_q12 = [(1,    "Alice",   500.0),
             (None, "Bob",     300.0),
             (3,    "Charlie", 750.0),
             (None, "Diana",   200.0),
             (5,    "Eve",     900.0)]

df_q12 = spark.createDataFrame(data_q12, ["user_id", "name", "amount"])
df_q12.write.mode("overwrite").parquet(INPUT_PATH)

print("Sample Parquet input (with nulls):")
spark.read.parquet(INPUT_PATH).show()

# The required Spark commands
df_input = spark.read.parquet(INPUT_PATH)

df_clean = df_input.filter(F.col("user_id").isNotNull())

(df_clean.write
         .mode("overwrite")
         .option("header", "true")
         .csv(OUTPUT_PATH))

print("After filtering null user_id and saving as CSV:")
spark.read.option("header","true").csv(OUTPUT_PATH).show()
print(f"Output saved to: {OUTPUT_PATH}")


Sample Parquet input (with nulls):


+-------+-------+------+
|user_id|   name|amount|
+-------+-------+------+
|      1|  Alice| 500.0|
|   NULL|    Bob| 300.0|
|      3|Charlie| 750.0|
|   NULL|  Diana| 200.0|
|      5|    Eve| 900.0|
+-------+-------+------+



After filtering null user_id and saving as CSV:


+-------+-------+------+
|user_id|   name|amount|
+-------+-------+------+
|      1|  Alice| 500.0|
|      3|Charlie| 750.0|
|      5|    Eve| 900.0|
+-------+-------+------+

Output saved to: /tmp/path_to_output


---
## Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

Both modes refer to **where the Driver process runs** when submitting a Spark application to a cluster.

| | **Client Mode** | **Cluster Mode** |
|---|---|---|
| **Driver location** | Runs on the **machine that submits the job** (your laptop / edge node) | Runs on **one of the worker nodes** inside the cluster |
| **Network dependency** | Driver must stay connected to cluster throughout job; network interruption kills the job | Driver is inside the cluster — no dependency on submitting machine |
| **Use case** | Interactive development, Jupyter notebooks, debugging | Production jobs, long-running batch pipelines |
| **Log access** | Driver logs visible directly in the terminal | Driver logs must be accessed from cluster UI or log aggregator |
| **Resource usage** | Submitting machine consumes Driver memory/CPU | Cluster handles all resource allocation |

**Simple analogy:**
- **Client Mode** → You manage the project from your desk (submit machine). If you leave, the project stops.
- **Cluster Mode** → You hand the project to the cluster. It runs independently; you can disconnect.

```bash
# Client Mode (default for spark-shell and pyspark)
spark-submit --master yarn --deploy-mode client app.py

# Cluster Mode (for production)
spark-submit --master yarn --deploy-mode cluster app.py
```


---
## Q14: Write a query to filter a dataset for rows where the `region` is 'North' OR the `priority` is 'High'.


In [12]:
# Q14 — Filter: region = 'North' OR priority = 'High'

data_q14 = [(1, "North",  "High",   5000.0),
             (2, "South",  "Low",    1200.0),
             (3, "East",   "High",   3400.0),
             (4, "West",   "Medium", 800.0),
             (5, "North",  "Low",    2100.0),
             (6, "Central","High",   4500.0),
             (7, "South",  "Medium", 950.0)]

df_q14 = spark.createDataFrame(data_q14, ["order_id", "region", "priority", "sales"])

print("Full dataset:")
df_q14.show()

# Filter: region = 'North' OR priority = 'High'
result_q14 = df_q14.filter(
    (F.col("region") == "North") | (F.col("priority") == "High")
)

print("Filtered — region='North' OR priority='High':")
result_q14.show()


Full dataset:


+--------+-------+--------+------+
|order_id| region|priority| sales|
+--------+-------+--------+------+
|       1|  North|    High|5000.0|
|       2|  South|     Low|1200.0|
|       3|   East|    High|3400.0|
|       4|   West|  Medium| 800.0|
|       5|  North|     Low|2100.0|
|       6|Central|    High|4500.0|
|       7|  South|  Medium| 950.0|
+--------+-------+--------+------+

Filtered — region='North' OR priority='High':


+--------+-------+--------+------+
|order_id| region|priority| sales|
+--------+-------+--------+------+
|       1|  North|    High|5000.0|
|       3|   East|    High|3400.0|
|       5|  North|     Low|2100.0|
|       6|Central|    High|4500.0|
+--------+-------+--------+------+



---
## Q15: When exploring a dataset, why is it safer to use `.show(5)` instead of `.collect()` on a multi-terabyte dataset?

**`.collect()`** retrieves **every single row** from all executors and brings them into the **Driver's memory** as a Python list.

On a multi-terabyte dataset:
- Terabytes of data are sent over the network to the Driver
- The Driver runs out of memory → **OutOfMemoryError (OOM)**
- The entire application crashes
- Even if it doesn't crash, it is extremely slow due to network transfer

**`.show(5)`** (or `.show(n)`) only fetches the **first n rows**:
- Spark stops scanning after finding 5 rows — it does not read the whole dataset
- Only those 5 rows are sent to the Driver — negligible memory usage
- Fast, safe, and suitable for quick data inspection

| | `.show(5)` | `.collect()` |
|---|---|---|
| **Data transferred to Driver** | Only 5 rows | ALL rows |
| **Risk of OOM** | None | Very high on large datasets |
| **Speed** | Near-instant | Extremely slow on TB data |
| **Use case** | Data exploration / debugging | Only safe on small DataFrames |

**Rule of thumb:** Never use `.collect()` in production or on large datasets. Use `.show()`, `.take(n)`, or `.first()` for inspection, and aggregations for analysis.


In [13]:
# Q15 — show() vs collect() demonstration

import time

# Create a moderately large DataFrame to illustrate
df_large = spark.range(0, 100000).withColumn(
    "value", F.rand() * 1000
)

# show(5) — safe, fast
t0 = time.time()
df_large.show(5)
show_time = time.time() - t0
print(f"show(5) completed in {show_time:.3f}s — fetched only 5 rows")

# collect() on the same data — brings ALL rows to driver
t0 = time.time()
rows = df_large.collect()   # safe here since 100K rows is small, but illustrates the risk
collect_time = time.time() - t0
print(f"collect() completed in {collect_time:.3f}s — fetched {len(rows):,} rows into driver memory")

print(f"\nOn a multi-TB dataset, collect() would attempt to bring")
print(f"billions of rows into the Driver → OutOfMemoryError (OOM)")
print(f"show(5) always fetches only the requested rows — always safe.")


+---+------------------+
| id|             value|
+---+------------------+
|  0| 755.2448339389313|
|  1| 847.7268051172434|
|  2| 323.5182831836991|
|  3| 291.5403890531754|
|  4|149.23391718030598|
+---+------------------+
only showing top 5 rows
show(5) completed in 0.276s — fetched only 5 rows


collect() completed in 2.229s — fetched 100,000 rows into driver memory

On a multi-TB dataset, collect() would attempt to bring
billions of rows into the Driver → OutOfMemoryError (OOM)
show(5) always fetches only the requested rows — always safe.


---
## Cleanup

In [14]:
spark.stop()
print("Spark session stopped cleanly.")
print("\nAll 15 questions answered successfully.")


Spark session stopped cleanly.

All 15 questions answered successfully.
